## Prerequisites

This notebook demonstrates how to invoke a **deployed real-time inference endpoint** on Azure for the SNOMED Clinical Terminology Mapper model. Follow the steps below in order before running this notebook.

### Deployment Sequence

1. **Acquire the Azure Application Offer** — Subscribe to the *SNOMED Clinical Terminology Mapper* offer in the [Azure Marketplace](https://azuremarketplace.microsoft.com) to obtain the Solution Template.
2. **Launch guided deployment** — From the Marketplace listing, start the guided deployment flow for the solution.
3. **Supply configuration** — Complete the guided UX forms (defined by the offer's `createUiDefinition`) with the required parameters.
4. **Deploy resources** — Confirm deployment; Azure provisions the resources defined in the offer's `mainTemplate.json`.
5. **Wait for deployment to succeed** — Verify the deployment reaches a *Succeeded* state before proceeding. Note the endpoint name from the deployment output.
6. **Run this notebook** — Execute the cells below against the real-time inference endpoint provisioned by the deployment.

### Environment

- **Notebook runtime**: This notebook is intended to run in **Azure Machine Learning workspace notebooks** (e.g. via an Azure ML Studio compute instance or Studio notebooks).
- **Jupyter kernel**: Use a kernel with the `azure.ai.ml` Python package installed, for example **Python 3.10 — SDK v2** (`python310-sdkv2`).
- **Authentication**: The kernel must be authenticated to Azure so that Azure ML SDK calls succeed (e.g. `DefaultAzureCredential()`). The authenticated identity must be granted Azure RBAC permissions appropriate for reading the workspace and online endpoint configuration, and for retrieving endpoint credentials. Consult your organization's access-policy guidance and the [Azure ML endpoint credential documentation](https://learn.microsoft.com/azure/machine-learning/how-to-authenticate-online-endpoint) for details.
- **Network access**: The notebook's compute instance must be able to reach the endpoint's `scoring_uri` (no outbound firewall or VNET restrictions that block it).


## SNOMED Clinical Terminology Mapper

- **Model**: [snomed_vdb_resolver](https://nlp.johnsnowlabs.com/2024/03/11/snomed_resolver_pipeline_en.html)

- **Model Description**: This pretrained pipeline extracts clinical entities from clinical text and maps them to their corresponding SNOMED codes.

In [2]:
ENDPOINT_NAME = "<CUSTOMER_TO_PUT_DEPLOYED_ENDPOINT_NAME>"

### Input Format

To use the model, you need to provide input in one of the following supported formats:

#### JSON Format

Provide input as JSON. We support two variations within this format:

1. **Array of Text Documents**:
   Use an array containing multiple text documents. Each element represents a separate text document.

   ```json
   {
       "text": [
           "Text document 1",
           "Text document 2",
           ...
       ]
   }
   ```

2. **Single Text Document**:
   Provide a single text document as a string.


   ```json
   {
       "text": "Single text document"
   }
   ```

#### JSON Lines (JSONL) Format

Provide input in JSON Lines format, where each line is a JSON object representing a text document.

```
{"text": "Text document 1"}
{"text": "Text document 2"}
```

### Authentication and Endpoint Lookup

The cell below uses `DefaultAzureCredential()` to authenticate to Azure, then retrieves the endpoint's scoring URI and primary access key from the workspace configuration. If you see authentication errors, verify that you are logged into Azure Cloud Shell / your compute instance and that your identity has sufficient role assignments on the workspace.


In [3]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import pandas as pd
import requests
import json

# Set display options
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


ml_client = MLClient.from_config(DefaultAzureCredential())
endpoint_url = ml_client.online_endpoints.get(ENDPOINT_NAME).scoring_uri
key = ml_client.online_endpoints.get_keys(ENDPOINT_NAME).primary_key


def invoke_realtime_endpoint(record, content_type="application/json", accept="application/json"):
    response = requests.post(
        endpoint_url,
        headers={
            "Authorization": f"Bearer {key}",
            "Content-Type": content_type,
        },
        data=json.dumps(record) if content_type == "application/json" else record,

    )

    response_body = response.content.decode("utf-8")

    if accept == "application/json":
        return json.loads(response_body)
    elif accept == "application/jsonlines":
        return response_body
    else:
        raise ValueError(f"Unsupported accept type: {accept}")

### Sample Data (Synthetic — for demonstration only)

The following cells define sample clinical text documents used in the examples below.

In [4]:
docs = [
    """The patient is a 40-year-old white male who presents with a chief complaint of "chest pain". The patient is diabetic and has a prior history of coronary artery disease. The patient presents today stating that his chest pain started yesterday evening and has been somewhat intermittent. He has been advised 81 milligrams QDay, 50 units in a.m., 50 mg QDay, 1/150 sublingually PRN chest pain.""",

    """Two weeks prior to presentation, she was treated with a five-day course of for a respiratory tract infection. She was on , , and for T2DM and and for HTG. She had been on for six months at the time of presentation. The patient was seen by the endocrinology service and she was discharged on 40 units of at night, 12 units of with meals, and 1000 mg two times a day."""
]

sample_text = """This is an 82-year-old male with a history of prior tobacco use, hypertension, chronic renal insufficiency, chronic obstructive pulmonary disease, gastritis, and transient ischemic attack. He initially presented to Braintree with ST elevation and was transferred to St. Margaret’s Center. He underwent cardiac catheterization because of the left main coronary artery stenosis, which was complicated by hypotension and bradycardia. He required atropine, IV fluids, and dopamine."""

### JSON

#### Example 1: single text document

In [5]:
input_json_data = {"text": sample_text}
response_json = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
pd.DataFrame(response_json["predictions"])

,begin,end,ner_chunk,ner_label,ner_confidence,concept_code,resolution,score,all_codes,domain_class,all_resolutions,all_score
0,52,58,tobacco,Smoking,0.9935,39953003,tobacco,1.000000,"[39953003, 66562002, 102407002, 333781000000109, 721131000000108]","[, , , , ]","[tobacco, cigarette smoking tobacco, tobacco smoke, tobacco industry, tobacco industry]","[1.0000004768371582, 0.9128881692886353, 0.9106559753417969, 0.8801187872886658, 0.8801187872886658]"
1,65,76,hypertension,Hypertension,0.9999,38341003,hypertension,1.000000,"[38341003, 250765005, 75367002, 73578008, 28119000]","[Clinical Finding, , , Morph Abnormality, ]","[hypertension, blood pressure, blood pressure, hyperdistension, renal hypertension]","[0.9999998211860657, 0.906682550907135, 0.906682550907135, 0.8890679478645325, 0.885756254196167]"
2,79,105,chronic renal insufficiency,Kidney_Disease,0.91976666,723190009,chronic renal insufficiency,1.000000,"[723190009, 425369003, 723188008, 73272005, 90688005]","[, Clinical Finding, , , ]","[chronic renal insufficiency, chronic progressive renal insufficiency, renal insufficiency, chronic insufficiency, chronic renal failure]","[1.0000001192092896, 0.923916757106781, 0.9123024940490723, 0.8887664079666138, 0.8822141885757446]"
3,108,144,chronic obstructive pulmonary disease,Disease_Syndrome_Disorder,0.5127,13645005,chronic obstructive pulmonary disease,1.000000,"[13645005, 155565006, 155569000, 195948000, 195935004]","[, , , , ]","[chronic obstructive pulmonary disease, chronic obstructive lung disease, chronic obstructive lung disease, chronic obstructive lung disease, chronic obstructive lung disease]","[0.9999999403953552, 0.9534465670585632, 0.9534465670585632, 0.9534465670585632, 0.9534465670585632]"
4,147,155,gastritis,Disease_Syndrome_Disorder,0.9959,4556007,gastritis,1.000000,"[4556007, 235653009, 155716003, 196741008, 723096000]","[, , , , Clinical Finding]","[gastritis, gastritis, helicobacter gastritis, helicobacter gastritis, bacterial gastritis]","[0.9999998807907104, 0.9999998807907104, 0.909252405166626, 0.909252405166626, 0.9042367339134216]"
5,162,186,transient ischemic attack,Cerebrovascular_Disease,0.5781,266257000,transient ischemic attack,1.000000,"[266257000, 313242003, 266257000, 57357009, 266257000]","[Clinical Finding, , Clinical Finding, , Clinical Finding]","[transient ischemic attack, transient ischemic attacks, transient ischaemic attack, transient ischemia, tia - transient ischemic attack]","[1.000000238418579, 0.9825674295425415, 0.9563201665878296, 0.9527279138565063, 0.9329274892807007]"
6,230,241,ST elevation,snomed_term,0.64225,164931005,st elevation,1.000000,"[164931005, 142094006, 76388001, 103720008, 103720008]","[, , , , Procedure]","[st elevation, st elevation, st segment elevation, elevation, elevation, nos]","[1.000000238418579, 1.000000238418579, 0.887158215045929, 0.831535279750824, 0.813231348991394]"
7,302,324,cardiac catheterization,Procedure,0.7781,141945006,cardiac catheterization,1.000000,"[141945006, 41976001, 41976001, 41976001, 45211000]","[, , Procedure, Procedure, ]","[cardiac catheterization, cardiac catheterization, cardiac catheterization procedure, cardiac catheterisation, catheterization]","[1.0000001192092896, 1.0000001192092896, 0.9511891603469849, 0.9394109845161438, 0.9253652095794678]"
8,337,374,the left main coronary artery stenosis,PROBLEM,0.84225005,371804009,left main coronary artery disease,0.916886,"[371804009, 876857001, 1255569006, 1255569006, 1255569006]","[, , No_Concept_Class, No_Concept_Class, ]","[left main coronary artery disease, stenosis of left coronary artery main stem, ostial left main coronary artery stenosis, stenosis of ostial left main coronary artery, stenosis of ostium of left main coronary artery]","[0.9168860912322998, 0.910420835018158, 0.8989304304122925, 0.8936088681221008, 0.883314311504364]"
9,402,412,hypotension,VS_Finding,0.9877,155487000,hypotension,1.000000,"[155487000, 45007003, 28651003, 77545000, 45007003]","[, Clinical 

#### Example 2: array of text documents

In [6]:
input_json_data = {"text": docs}
response_json = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
response_json

{'predictions': [[{'begin': 80,
    'end': 89,
    'ner_chunk': 'chest pain',
    'ner_label': 'Symptom',
    'ner_confidence': '0.7448',
    'concept_code': '139228007',
    'resolution': 'chest pain',
    'score': 1.000000238418579,
    'all_codes': ['139228007',
     '29857009',
     '239671000000106',
     '426396005',
     '498151000000109'],
    'domain_class': ['', '', '', '', ''],
    'all_resolutions': ['chest pain',
     'chest pain',
     'cardiac chest pain',
     'cardiac chest pain',
     '[d]chest pain'],
    'all_score': [1.000000238418579,
     1.000000238418579,
     0.9261189699172974,
     0.9261189699172974,
     0.9083012342453003]},
   {'begin': 108,
    'end': 115,
    'ner_chunk': 'diabetic',
    'ner_label': 'Diabetes',
    'ner_confidence': '0.9994',
    'concept_code': '280137006',
    'resolution': 'diabetic foot',
    'score': 0.8424347639083862,
    'all_codes': ['280137006',
     '160670007',
     '138058003',
     '385804009',
     '105452009'],
    'do

### JSON Lines

In [7]:
def create_jsonl(records):
    if isinstance(records, str):
        records = [records]
    json_records = [{"text": text} for text in records]
    json_lines = "\n".join(json.dumps(record) for record in json_records)
    return json_lines

In [8]:
input_jsonl_data = create_jsonl(docs)
data = invoke_realtime_endpoint(input_jsonl_data, content_type="application/jsonlines" , accept="application/jsonlines" )
print(data)

{"predictions": [{"begin": 80, "end": 89, "ner_chunk": "chest pain", "ner_label": "Symptom", "ner_confidence": "0.7448", "concept_code": "139228007", "resolution": "chest pain", "score": 1.000000238418579, "all_codes": ["139228007", "29857009", "239671000000106", "426396005", "498151000000109"], "domain_class": ["", "", "", "", ""], "all_resolutions": ["chest pain", "chest pain", "cardiac chest pain", "cardiac chest pain", "[d]chest pain"], "all_score": [1.000000238418579, 1.000000238418579, 0.9261189699172974, 0.9261189699172974, 0.9083012342453003]}, {"begin": 108, "end": 115, "ner_chunk": "diabetic", "ner_label": "Diabetes", "ner_confidence": "0.9994", "concept_code": "280137006", "resolution": "diabetic foot", "score": 0.8424347639083862, "all_codes": ["280137006", "160670007", "138058003", "385804009", "105452009"], "domain_class": ["", "", "", "", ""], "all_resolutions": ["diabetic foot", "diabetic diet", "diabetic diet", "diabetic care", "diabetic relative"], "all_score": [0.842